# 3. BM25 + Gemini query expansion

Highest completed recall among these train versions; first-use latency and abstention limitations.

This is a recorded full-train comparison, not a new evaluation.

In [1]:
from pathlib import Path
import json,sys
import pandas as pd
ROOT=next(p for p in [Path.cwd().resolve(),*Path.cwd().resolve().parents] if (p/'src/retrieval.py').exists())
sys.path.insert(0,str(ROOT/'src'))
sys.path.insert(0,str(ROOT/'scripts'))
from presentation_review import selection,cached_demo
model=selection(3)
model['config_snapshot']

{'name': 'llm_gemini_rewrite',
 'provider': 'gemini',
 'model': 'gemini-3.5-flash-lite',
 'model_revision': 'stable identifier; actual modelVersion recorded in every response',
 'runtime': 'gemini-rest-v1beta',
 'prompt_file': 'prompts/retrieval_rewrite_v1.txt',
 'temperature': 0,
 'max_output_tokens': 256,
 'thinking_level': 'minimal',
 'baseline_config': 'configs/bm25_focused_stemmed.json',
 'expose_in_demo': True}

In [2]:
m=model['train_metrics']
table=pd.DataFrame(m['retrieval'])[['k','n_queries','precision','recall','hit']]
table

,k,n_queries,precision,recall,hit
0,1,140,0.216779,0.529522,0.635714
1,3,140,0.122140,0.711065,0.785714
2,5,140,0.084691,0.790986,0.871429
3,10,140,0.052401,0.878983,0.928571


In [3]:
print('Correct negative abstention:',m['negative_counts'])
print('False answerable abstention:',m['false_abstention'])
print('Cached latency:',m['latency_ms'])

Correct negative abstention: {'n': 23, 'correct': 0}
False answerable abstention: 0.0
Cached latency: {'median': 12.404250039253384, 'p95': 15.954549965681505}


## Cached example

The question comes from recorded train predictions. Config settings and original source offsets remain visible; the model cannot call a hosted provider here.

In [4]:
demo=cached_demo(model)
print(demo['query'])
print(demo.get('summary') or demo['message'])
pd.DataFrame(demo['hits'])[['file_path','start','end','text']]

Consider the Amendment No. 3 to Wireless Content License Agreement between Fox Mobile Entertainment, Inc. and Glu Mobile Inc.; What is the expiration date of this contract?
3 passages retrieved.


,file_path,start,end,text
0,cuad/GluMobileInc_20070319_S-1A_EX-10.09_43663...,0,1000,"AMENDMENT NO. 3\n\nDated as of February 19, 20..."
1,cuad/GluMobileInc_20070319_S-1A_EX-10.09_43663...,1600,1902,"x Mobile Entertainment, Inc. (""Fox"")\n\n\n\nBy..."
2,cuad/GluMobileInc_20070319_S-1A_EX-10.09_43663...,800,1800,"dom of Heaven, Mr. and Mrs. Smith, In Her Shoe..."


Full code walkthrough: ../experiments/10_gemini_bm25.ipynb. Historical acceptance and current preference are separate; see REPORT.md for limitations.